# Chapter 6 · Talking to your model

Your tinyGPT scored 1.49. What does that look like as writing? In this chapter you open
the box that writes, `generate`, and turn its two dials: **temperature** (how adventurous
it is) and **top-k** (how many candidates it considers). Then two honest questions. Do
the settings that make the model *look* best also make it *score* best? And is it just
copying Shakespeare?

| | |
|---|---|
| **Open box** | the writing loop: `generate` in `tinygpt/sampling.py` |
| **The number** | 1.49, and what temperature does to it |
| **Time** | about an hour |
| **Python you need** | loops, functions; a little PyTorch |
| **GPU** | not needed if you kept your model in chapter 5; needed if it has to be trained again |

**How this chapter works.** Five steps, in order:

1. **Predict** `everyone`: guess before you run anything. A wrong guess is useful: it shows you what you expected.
2. **Run** `everyone`: run the cells and compare with your guesses.
3. **Investigate** `everyone`: open the box. Read the code that just ran and answer a few questions.
4. **Modify** `most people`: change one thing, predict what will happen, then explain what did.
5. **Make** `optional challenge`: build this chapter's piece yourself. A check tells you when it's right.

Run a cell with **Shift+Enter**. If the notebook restarts, run the setup cell again first.

## Setup

In [ ]:
#@title Setup: run this cell first (about 30 seconds) { display-mode: "form" }
# Downloads tinyGPT (on Colab) and installs it. You don't need to read or change this cell.
import importlib, os, subprocess, sys
from pathlib import Path

def run(*command):
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"{' '.join(command)} failed:\n{done.stdout}{done.stderr}")

if "google.colab" in sys.modules:
    root = Path("/content/tinyGPT")
    if not root.exists():
        run("git", "clone", "-q", "https://github.com/cgpan/tinyGPT.git", str(root))
else:
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "src" / "tinygpt").is_dir()), None)
    if root is None:
        raise FileNotFoundError("Can't find the tinyGPT folder. Open this notebook from inside it, or use Colab.")
os.chdir(root)
run(sys.executable, "-m", "pip", "install", "-q", ".")
importlib.invalidate_caches()

import tinygpt
tinygpt.print_setup()

### Your model from chapter 5

This loads the model you kept at the end of chapter 5: from your Google Drive on Colab
(Colab may ask for permission again), from the `runs/` folder on your own computer. If
there is no kept model, it trains one first, which takes 6 to 8 minutes on a T4.

In [ ]:
import math
import torch

model, tok, history = tinygpt.load_kept_model()
_, train_ids, val_ids = tinygpt.load_data()
text = tinygpt.load_text()
train_text = text[: len(train_ids)]          # the part of the text the model trained on
device = next(model.parameters()).device

## 1 · Predict `everyone`

*Temperature* scales how bold the model is when it picks each character. At 1.0 it uses
its own probabilities. Below 1 it plays safe and almost always picks its favourite;
above 1 it takes chances on unlikely characters.

In [ ]:
guess_cold = None             # at temperature 0.1, what will its writing look like? (in words)
guess_hot = None              # at temperature 2.0?
guess_best_temperature = None # which gives the lowest validation loss: 0.5, 1.0 or 2.0?
guess_longest_copy = None     # in 1,000 characters it writes, the longest piece copied
                              # character for character from Shakespeare: how many characters?

## 2 · Run `everyone`

Your model, writing at temperature 1.0:

In [ ]:
print(tinygpt.sample(model, tok, prompt="ROMEO:", length=400, seed=0))

Now give it your own start. Any of the 65 characters will do.

In [ ]:
print(tinygpt.sample(model, tok, prompt="JULIET:\nO ", length=300, seed=1))

The same start at five temperatures:

In [ ]:
for temperature in [0.1, 0.5, 1.0, 1.5, 2.0]:
    print(f"----- temperature {temperature} -----")
    print(tinygpt.sample(model, tok, prompt="ROMEO:", length=200, temperature=temperature, seed=0))

How close were your first two guesses?

*Your explanation (double-click to edit):*

## 3 · Investigate `everyone`

### Open the box

In [ ]:
tinygpt.show(tinygpt.generate)

1. `context = idx[:, -model.cfg.block_size:]`: why give the model only the last
   `block_size` characters? (Chapter 4 has the answer.)
2. `logits[:, -1, :]`: the model scores a next character at *every* position. Why keep
   only the last one?
3. `/ temperature`: what does dividing the scores by a small number do? By a big one?
   The next cell shows it on three made-up scores.
4. `torch.multinomial` rolls weighted dice: likely characters come up often, unlikely
   ones sometimes. What would always picking the likeliest do instead? (You'll try it in
   step 4.)
5. `torch.cat` adds the new character to the end. Why does writing take longer the
   more characters you ask for, one at a time?

In [ ]:
scores = torch.tensor([2.0, 1.0, 0.0])          # three candidate characters
for temperature in [0.1, 0.5, 1.0, 2.0]:
    print(f"temperature {temperature}: probabilities {torch.softmax(scores / temperature, dim=0).numpy().round(3)}")

*Your explanation (double-click to edit):*

### What does 1.49 mean?

*Perplexity*, e to the power of the loss, turns the number into "on average, as unsure
as picking among this many characters":

In [ ]:
loss = history[-1]["val_loss"]
print(f"knowing nothing: 65.0   counting pairs: {math.exp(2.4819):.1f}   your model: {math.exp(loss):.1f}")

### Which temperature scores best?

Temperature changes the probabilities, so it changes the loss too. This measures the
loss at several temperatures, on validation text and on training text. **Check your
third guess.**

In [ ]:
from torch.nn import functional as F

def loss_at(temperature, ids, batches=10):
    torch.manual_seed(0)                                   # the same batches for every temperature
    total = 0.0
    for _ in range(batches):
        x, y = tinygpt.get_batch(ids, model.cfg.block_size, 64, device)
        with torch.no_grad():
            logits, _ = model(x)
        total += F.cross_entropy((logits / temperature).flatten(0, 1), y.flatten()).item()
    return total / batches

temperatures = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0]
on_validation = {t: loss_at(t, val_ids) for t in temperatures}
for t in temperatures:
    print(f"temperature {t}:  validation {on_validation[t]:.3f}   training {loss_at(t, train_ids):.3f}")

On training text, temperature 1.0 scores best: the model's own probabilities fit the
text it learned from. On validation text, a temperature a little *above* 1 scores best.
On new text the model is **overconfident**: it bets too heavily on its favourites,
because it partly memorised its training text (the gap between the two curves in
chapter 5). A slightly higher temperature softens those bets.

Low temperatures do the opposite: their writing often *looks* smoother, but they score
worse, because they hide the model's uncertainty. Whenever you show someone a sample,
say which temperature made it.

*Your explanation (double-click to edit):*

### Is it just copying Shakespeare?

`longest_copy` finds the longest piece of the model's writing that appears, character
for character, somewhere in the training text. **Check your fourth guess.**

In [ ]:
written = tinygpt.sample(model, tok, prompt="\n", length=1000, seed=0)
copy = tinygpt.longest_copy(written, train_text)
print(f"longest copied piece: {len(copy)} characters: {copy!r}")

What kind of text are the longest copies: whole lines of verse, or names and common
phrases? Would you call this model a plagiarist?

*Your explanation (double-click to edit):*

## 4 · Modify `most people`

**A. Always pick the likeliest.** `top_k=1` keeps only the single best candidate, so
there is no dice roll at all. **Predict** what happens over 300 characters.

In [ ]:
print(tinygpt.sample(model, tok, prompt="ROMEO:", length=300, top_k=1))

Why does it get stuck? What does the dice roll add?

*Your explanation (double-click to edit):*

**B. Fewer candidates.** `top_k=5` lets the model choose only among its five favourites
at each step. Compare with the temperature 1.0 sample in step 2.

In [ ]:
print(tinygpt.sample(model, tok, prompt="ROMEO:", length=300, top_k=5, seed=0))

**C. Copying at a lower temperature.** **Predict:** at temperature 0.5, will the longest
copied piece be longer or shorter?

In [ ]:
cautious = tinygpt.sample(model, tok, prompt="\n", length=1000, temperature=0.5, seed=0)
copy = tinygpt.longest_copy(cautious, train_text)
print(f"temperature 0.5, longest copied piece: {len(copy)} characters: {copy!r}")

*Your explanation (double-click to edit):*

## 5 · Make `optional challenge`

Write the generation loop yourself, in its simplest form: at each step, pick the single
likeliest next character. The check compares your loop's output with tinyGPT's
`generate(..., top_k=1)`, on a sequence longer than the model's memory.

In [ ]:
def my_greedy(model, idx, max_new_tokens):
    """Add max_new_tokens characters to idx, each time the likeliest next one."""
    for _ in range(max_new_tokens):
        context = idx                 # 1. the model can only read the last block_size characters
        logits, _ = model(context)    # 2. scores for every position
        scores = logits               # 3. keep only the scores for the next character
        next_id = scores              # 4. pick the likeliest: torch.argmax(..., dim=-1, keepdim=True)
        idx = idx                     # 5. add it to the end with torch.cat([...], dim=1)
    return idx

tinygpt.check.greedy(my_greedy)

When it passes, let your loop write with your model:

In [ ]:
start = torch.tensor([tok.encode("ROMEO:")], device=device)
with torch.no_grad():
    print(tok.decode(my_greedy(model, start, 200)[0].tolist()))

Going further: add a `temperature` argument, and roll the dice with `torch.multinomial` instead of `argmax`.

**Stuck?** Try first, then compare with [the solution](https://github.com/cgpan/tinyGPT/blob/main/solutions/06_talking.md) (also in the `solutions/` folder of your copy).

## Where the number stands

In [ ]:
best = min(on_validation, key=on_validation.get)
tinygpt.scoreboard({"yours, temperature 1.0": on_validation[1.0],
                    f"yours, temperature {best}": on_validation[best]})

## Common questions

**Why is it called temperature?** The name comes from physics: in a hot gas, particles
take unlikely states more often. A hot model picks unlikely characters more often.

**Which temperature should I use?** To measure the model, 1.0, or whatever scores best on
validation text, and say which. For writing to read, 0.7 to 1.0 usually looks best. Below
that it starts to repeat itself.

**Do ChatGPT and Claude use temperature?** Yes, along with similar dials. Unlike
tinyGPT, they are also trained further to follow instructions. tinyGPT only continues
text in the style it learned.

**Can I ask it a question?** You can type one as a prompt, but it will just continue it
like a line from a play. It has never seen a question being answered.

**Why is it overconfident on new text?** It memorised parts of its 1 million training
characters. More training text would help more than anything else; dropout helps a little.

## If something breaks

| What you see | What to do |
|---|---|
| Google Drive asks for permission | Allow it, with the Google account you used in chapter 5. |
| `No model found ... training one now` | Your kept model wasn't found. It trains one (6 to 8 minutes on a T4) and keeps it for next time. |
| `ValueError: ... is not in the vocabulary` | Your prompt has a character outside the 65: an accent, an emoji, or a digit other than 3. |
| Writing is slow | It writes one character at a time: a few seconds per 300 characters on a CPU. |

## Next

**Chapter 7 · How do you know it works?** Tests that catch a broken model before you
waste an hour training it.